In [1]:
!pip install torch pandas scikit-learn

In [2]:
import pandas as pd
import random

def generate_dna(length):
    return ''.join(random.choices(['A','T','C','G'], k=length))

data = []

for _ in range(1000):
    seq = generate_dna(50)

    # Simple rule: sequences with many "TATA" motifs → promoter
    label = 1 if "TATA" in seq else 0

    data.append({"sequence": seq, "label": label})

df = pd.DataFrame(data)
df.head()

,sequence,label
0,ACGAAAACGGCAATTATTTAAAGAGGATGAGCCGGTCGCTATGGGT...,0
1,ATAAAGGCAGAGGGATGAAATGGCCTAAGGTCTGCACCGTATGGGT...,0
2,AGCTGATAAGATGGATCTAGTTGCGTTCATCGGCCTAAAGTGTGCC...,0
3,TCTTTACTGCCCTGTGTTCGATGGATTCCAGCCGCCATAGAGTGTT...,0
4,ATTGCCCTTGAGCAACGCTTGATTCACTACGCAGTATCTAGCCAGG...,0


In [3]:
import numpy as np

def encode_sequence(seq):
    mapping = {'A':[1,0,0,0],
               'T':[0,1,0,0],
               'C':[0,0,1,0],
               'G':[0,0,0,1]}

    return np.array([mapping[base] for base in seq])

X = np.array([encode_sequence(seq) for seq in df["sequence"]])
y = df["label"].values

X.shape

(1000, 50, 4)

In [4]:
import torch
import torch.nn as nn

class DNAClassifier(nn.Module):
    def __init__(self):
        super(DNAClassifier, self).__init__()

        self.conv1 = nn.Conv1d(4, 16, kernel_size=3)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(16*24, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)  # reshape for Conv1D
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

model = DNAClassifier()
model

DNAClassifier(
  (conv1): Conv1d(4, 16, kernel_size=(3,), stride=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=384, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(5):
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = loss_fn(outputs, y_train)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.6705488562583923
Epoch 2, Loss: 0.6453406810760498
Epoch 3, Loss: 0.6213927268981934
Epoch 4, Loss: 0.5980861783027649
Epoch 5, Loss: 0.575551450252533


In [6]:
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

with torch.no_grad():
    preds = model(X_test)
    preds = (preds > 0.5).float()

accuracy = (preds == y_test).float().mean()
print("Accuracy:", accuracy.item())

Accuracy: 0.824999988079071


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = preds.numpy()
y_true = y_test.numpy()

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

[[165   0]
 [ 35   0]]
              precision    recall  f1-score   support

         0.0       0.82      1.00      0.90       165
         1.0       0.00      0.00      0.00        35

    accuracy                           0.82       200
   macro avg       0.41      0.50      0.45       200
weighted avg       0.68      0.82      0.75       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
def predict_sequence(seq):
    encoded = encode_sequence(seq)
    tensor = torch.tensor(encoded, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        pred = model(tensor)

    return "Promoter" if pred.item() > 0.5 else "Non-Promoter"

# Test it
test_seq = "TATAGCGTATAGCGTATAGCGTATAGCGTATAGCGTATAGCGTATAGCGT"
print(predict_sequence(test_seq))

Non-Promoter


In [9]:
def predict_with_confidence(seq):
    encoded = encode_sequence(seq)
    tensor = torch.tensor(encoded, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        pred = model(tensor)

    prob = pred.item()

    return {
        "prediction": "Promoter" if prob > 0.5 else "Non-Promoter",
        "confidence": prob
    }

predict_with_confidence(test_seq)

{'prediction': 'Non-Promoter', 'confidence': 0.39596301317214966}

In [10]:
torch.save(model.state_dict(), "dna_model.pth")

In [1]:
import os
os.listdir()

['.config', 'sample_data']

In [4]:
import torch
import torch.nn as nn

class DNAClassifier(nn.Module):
    def __init__(self):
        super(DNAClassifier, self).__init__()

        self.conv1 = nn.Conv1d(4, 16, kernel_size=3)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(16*24, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

model = DNAClassifier()

In [6]:
import pandas as pd
import numpy as np
import random

def generate_dna(length):
    return ''.join(random.choices(['A','T','C','G'], k=length))

data = []
for _ in range(500):  # smaller = faster
    seq = generate_dna(50)
    label = 1 if "TATA" in seq else 0
    data.append({"sequence": seq, "label": label})

df = pd.DataFrame(data)

def encode_sequence(seq):
    mapping = {'A':[1,0,0,0],
               'T':[0,1,0,0],
               'C':[0,0,1,0],
               'G':[0,0,0,1]}
    return np.array([mapping[base] for base in seq])

X = np.array([encode_sequence(seq) for seq in df["sequence"]])
y = df["label"].values

In [7]:
from sklearn.model_selection import train_test_split
import torch

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

In [8]:
import torch.nn as nn

class DNAClassifier(nn.Module):
    def __init__(self):
        super(DNAClassifier, self).__init__()
        self.conv1 = nn.Conv1d(4, 16, kernel_size=3)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(16*24, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

model = DNAClassifier()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(2):  # quick training
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = loss_fn(outputs, y_train)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

# SAVE MODEL
torch.save(model.state_dict(), "dna_model.pth")

Epoch 1, Loss: 0.6615216732025146
Epoch 2, Loss: 0.6334264278411865


In [11]:
from google.colab import files
files.download("dna_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>